# 🗂️ Генератор датасета
Берёт фразы из `phrases.txt`, генерирует вопросы через LM Studio и сохраняет `dataset.jsonl`.

## ⚙️ Настройки

In [1]:
PHRASES_FILE   = "phrases.txt"
DATASET_FILE   = "dataset.jsonl"
LM_STUDIO_URL  = "http://localhost:1234/v1"   # адрес LM Studio
MODEL_NAME     = "qwen/qwen3.5-9b"            # имя модели из LM Studio

CHARACTER_SYSTEM_PROMPT = """Ты — Сидорович, торговец из игры S.T.A.L.K.E.R. Ты сидишь в своём бункере на Кордоне и торгуешь с сталкерами — покупаешь артефакты, продаёшь снаряжение, даёшь задания. Говоришь покровительственно, думаешь о выгоде, знаешь всё что происходит в Зоне."""

QUESTION_GEN_PROMPT = """Ты помогаешь создать датасет для обучения языковой модели.

Тебе дана фраза, которую говорит персонаж Сидорович (торговец из S.T.A.L.K.E.R.).
Придумай короткий вопрос или реплику сталкера, на которую Сидорович мог бы ответить именно этой фразой.
Вопрос должен быть естественным и логично вести к данному ответу.

Отвечай ТОЛЬКО вопросом, без объяснений и кавычек.

Фраза Сидоровича: {phrase}"""

print("Настройки загружены ✓")

Настройки загружены ✓


## 📦 Импорты

In [2]:
import json
import os
from openai import OpenAI

client = OpenAI(base_url=LM_STUDIO_URL, api_key="lm-studio")
print("OpenAI клиент создан ✓")

OpenAI клиент создан ✓


## 🔧 Вспомогательные функции

In [3]:
def load_phrases():
    if not os.path.exists(PHRASES_FILE):
        print(f"Файл {PHRASES_FILE} не найден. Сначала запусти collect_phrases.ipynb")
        return []
    with open(PHRASES_FILE, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def load_existing_answers():
    existing = set()
    if not os.path.exists(DATASET_FILE):
        return existing
    with open(DATASET_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                entry = json.loads(line)
                for msg in entry.get("messages", []):
                    if msg["role"] == "assistant":
                        existing.add(msg["content"])
            except Exception:
                pass
    return existing

def generate_question(phrase):
    prompt = QUESTION_GEN_PROMPT.format(phrase=phrase)
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.9,
            max_tokens=100,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"  [ОШИБКА] {e}")
        return None

def save_entry(question, answer):
    entry = {
        "messages": [
            {"role": "system",    "content": CHARACTER_SYSTEM_PROMPT},
            {"role": "user",      "content": question},
            {"role": "assistant", "content": answer},
        ]
    }
    with open(DATASET_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("Функции определены ✓")

Функции определены ✓


## ▶️ Запуск генерации
> ⚠️ LM Studio должен быть запущен с загруженной моделью!

In [4]:
phrases  = load_phrases()
existing = load_existing_answers()
new_phrases = [p for p in phrases if p not in existing]

print(f"Всего фраз:        {len(phrases)}")
print(f"Уже в датасете:    {len(existing)}")
print(f"Нужно обработать:  {len(new_phrases)}")

if not new_phrases:
    print("\nВсё уже обработано! Переходи к finetune.ipynb")

Всего фраз:        153
Уже в датасете:    0
Нужно обработать:  153


In [5]:
success = 0

for i, phrase in enumerate(new_phrases, 1):
    print(f"[{i}/{len(new_phrases)}] {phrase[:70]}...")
    question = generate_question(phrase)
    if question:
        save_entry(question, phrase)
        print(f"  Q: {question}")
        print(f"  A: {phrase[:70]}\n")
        success += 1
    else:
        print(f"  [пропускаю]\n")

total = len(existing) + success
print(f"\n✅ Готово! Добавлено: {success}, всего в датасете: {total}")
print(f"Файл: {DATASET_FILE}")

[1/153] # ── ОРИГИНАЛЬНЫЕ ФРАЗЫ ИЗ ИГРЫ ──────────────────────────────────────...
  Q: Какой смысл в таких делах?
  A: # ── ОРИГИНАЛЬНЫЕ ФРАЗЫ ИЗ ИГРЫ ──────────────────────────────────────

[2/153] Хабар принёс?...
  Q: Привет, Сидорович, ты уже вернулся из Зоны, хабар принёс?
  A: Хабар принёс?

[3/153] Есть что-то стоящее?...
  Q: А вот и товар, Сидорович. А есть что-то стоящее?
  A: Есть что-то стоящее?

[4/153] Что притащил?...
  Q: Сидорович, ты уже вернулся с рынка?
  A: Что притащил?

[5/153] Как дела, сталкер?...
  Q: Слушай, а как ты сам?
  A: Как дела, сталкер?

[6/153] Иди, не мозоль глаза....
  Q: Сколько стоит этот ящик с патронами?
  A: Иди, не мозоль глаза.

[7/153] Не стой как истукан....
  Q: Почему не двигаешься?
  A: Не стой как истукан.

[8/153] Ты что, пьян?...
  Q: Сидорович, ты точно меня не перепутал?
  A: Ты что, пьян?

[9/153] Хорошо, ну будет ещё, заходи....
  Q: Можно купить еще что-нибудь?
  A: Хорошо, ну будет ещё, заходи.

[10/153] Молодец, я доволен....

## 👀 Просмотр датасета (первые N записей)

In [6]:
PREVIEW_N = 5

with open(DATASET_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"Всего записей: {len(lines)}\n")
for line in lines[:PREVIEW_N]:
    entry = json.loads(line)
    msgs  = entry["messages"]
    print(f"USER : {msgs[1]['content']}")
    print(f"BOT  : {msgs[2]['content']}")
    print("-" * 60)

Всего записей: 153

USER : Какой смысл в таких делах?
BOT  : # ── ОРИГИНАЛЬНЫЕ ФРАЗЫ ИЗ ИГРЫ ──────────────────────────────────────────────
------------------------------------------------------------
USER : Привет, Сидорович, ты уже вернулся из Зоны, хабар принёс?
BOT  : Хабар принёс?
------------------------------------------------------------
USER : А вот и товар, Сидорович. А есть что-то стоящее?
BOT  : Есть что-то стоящее?
------------------------------------------------------------
USER : Сидорович, ты уже вернулся с рынка?
BOT  : Что притащил?
------------------------------------------------------------
USER : Слушай, а как ты сам?
BOT  : Как дела, сталкер?
------------------------------------------------------------
